# COURSE PROJECT: PROGRAMMING FOUNDATIONS FOR DATA ANALYSIS AND VISUALIZATION
**Semester 261 – Academic Year 2026–2027** | **Instructor:** Lê Thành Sách

---
## PART 3: TIME SERIES DATA
### Topic: 24-Hour-Ahead PM2.5 Forecasting — Hanoi vs Ho Chi Minh City

- **Group:** IndexError
- **Member in charge:** Nguyễn Văn Hoàng Khang – 2570754

**Problem.** At time $t$, using the PM2.5 history and the current weather, forecast the PM2.5 concentration
at time $t + 24h$ (the same hour tomorrow). This is a **time series regression** problem.

**Why compare two cities?** Hanoi (four seasons, winter temperature inversion) and Ho Chi Minh City (HCMC; wet/dry seasons)
have different pollution mechanisms. We apply **the same pipeline** to each city **independently** (no merging) to answer:
*how much does machine learning improve over a naive baseline in each city, and which city is harder to forecast?*

## 0. ENVIRONMENT SETUP

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
HORIZON = 24            # forecast 24 hours ahead
WHO_24H = 15            # WHO 2021 guideline for 24-hour mean PM2.5 (µg/m³)
CITY_COLORS = {'Hanoi': '#d62728', 'HCMC': '#1f77b4'}
import sklearn
print(f'Environment ready! (scikit-learn {sklearn.__version__}, pandas {pd.__version__})')

## 1. DATASET DESCRIPTION

### 1.1. Data sources (self-collected)
The data was **collected through public APIs** with the script `scripts/crawl_air_quality.py`:

| Component | Source | Notes |
|---|---|---|
| PM2.5 – Hanoi | [OpenAQ v3](https://openaq.org) – station *Công viên Nhân Chính – Khuất Duy Tiến* (Hanoi Air Quality Monitoring Network, sensor 13502151) | Reference monitor of the city network |
| PM2.5 – HCMC | OpenAQ v3 – station *CMT8* (AirGradient, sensor 11357424) | Low-cost sensor (the only HCMC station with continuous data in the period) |
| Weather | [Open-Meteo Historical Weather API](https://open-meteo.com) | Temperature, humidity, precipitation, wind, pressure, cloud cover, at the coordinates of each PM2.5 station |

- **PM2.5 values** are OpenAQ hourly means of the station's sub-hourly readings; each timestamp is the **start** of the hour, Vietnam local time (UTC+7).
- **Period:** the same calendar period for both cities, from the first full day of the Hanoi station to the last full day before the crawl
  (exact dates and number of hours are printed in Section 1.2).
- A complete hourly time grid is built: hours with no measurement become `NaN`.

**How the stations were chosen.** The script `scripts/check_station_coverage.py` lists every OpenAQ PM2.5 sensor within 25 km of
each city centre that reported during the period and measures, over the period: `coverage` (share of hours with a value),
`worst_hour_cov` (lowest coverage among the 24 hours of the day), `worst_month_cov` (lowest coverage among calendar months) and
`longest_gap_h` (longest run of missing hours). `worst_hour_cov` matters as much as `coverage`: if a station misses mostly the
afternoon hours, the gaps are **not at random** (the missing hours differ systematically from the measured ones) and both the
daily cycle and the forecast errors would be biased. We use the station with the highest coverage in each city.

| Column | Type | Meaning |
|---|---|---|
| `datetime` | datetime | Timestamp (local time, start of the hour) |
| `city` | categorical | City |
| `temperature_2m` | numerical | Temperature at 2 m (°C) |
| `relative_humidity_2m` | numerical | Relative humidity (%) |
| `precipitation` | numerical | Precipitation (mm) |
| `wind_speed_10m` | numerical | Wind speed at 10 m (km/h) |
| `wind_direction_10m` | numerical (angle) | Wind direction (degrees) → converted to a **categorical** 8-sector feature |
| `surface_pressure` | numerical | Surface pressure (hPa) |
| `cloud_cover` | numerical | Cloud cover (%) |
| `pm25` | numerical | PM2.5 concentration (µg/m³) — **the variable to forecast** |
| `pm25_source` | categorical | PM2.5 source (`openaq`, or `open-meteo` if the crawler had to fall back) |
| `station` | categorical | Name of the PM2.5 station |

### 1.2. Loading the data
The CSV files are read locally; on Colab, if they are not found, they are downloaded from GitHub (**set `GITHUB_RAW_BASE`** to the group's repo).

In [ ]:
GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/perckgg/NTLT/main/03_TimeSeries/data/timeseries'
CITY_FILES = {'Hanoi': 'air_quality_hanoi.csv', 'HCMC': 'air_quality_hcmc.csv'}
LOCAL_DIRS = ['data/timeseries', '../data/timeseries', '/content/data/timeseries', '/content']


def load_csv(filename, parse_dates=('datetime',)):
    """Read a CSV from the first local folder that has it, otherwise from GitHub."""
    for d in LOCAL_DIRS:
        path = os.path.join(d, filename)
        if os.path.exists(path):
            return pd.read_csv(path, parse_dates=list(parse_dates))
    return pd.read_csv(f'{GITHUB_RAW_BASE}/{filename}', parse_dates=list(parse_dates))


raw = {city: load_csv(f).set_index('datetime').sort_index().assign(city=city)
       for city, f in CITY_FILES.items()}
for city, df in raw.items():
    print(f'{city}: {df.shape[0]:,} rows × {df.shape[1]} columns | '
          f'{df.index.min():%Y-%m-%d %H:%M} → {df.index.max():%Y-%m-%d %H:%M} '
          f'({(df.index.max() - df.index.min()).days + 1} days) | station: {", ".join(df.station.unique())} | '
          f'PM2.5 source: {", ".join(df.pm25_source.unique())}')

coverage = load_csv('station_coverage.csv', parse_dates=('first', 'last'))
print('\nCandidate stations over the same period (output of scripts/check_station_coverage.py):')
display(coverage)
raw['Hanoi'].head()

### 1.3. Descriptive statistics
The WHO guideline of 15 µg/m³ is defined for the **24-hour mean**, so it is compared with **daily means**, not with single hours.
A daily mean is computed only for days with at least 18 of 24 hours measured (75% completeness, the usual rule for daily averages);
other days are left out rather than averaged over a few hours.

In [ ]:
MIN_HOURS_PER_DAY = 18   # 75% of 24 hours


def daily_means(s, min_hours=MIN_HOURS_PER_DAY):
    """Daily mean of an hourly series; NaN for days with fewer than `min_hours` measured hours."""
    days = s.resample('D')
    return days.mean().where(days.count() >= min_hours)


raw['Hanoi'].info()
summary = pd.concat({city: df['pm25'].describe() for city, df in raw.items()}, axis=1).T
summary['missing_rate'] = [df['pm25'].isna().mean() for df in raw.values()]
daily = {city: daily_means(df['pm25']).dropna() for city, df in raw.items()}
summary['valid days'] = [len(d) for d in daily.values()]
summary['share of days > WHO 24h'] = [(d > WHO_24H).mean() for d in daily.values()]
display(summary.round(3))
display(pd.concat({city: df.describe().T for city, df in raw.items()}).round(2))

### 1.4. Time series visualization
**Figure 1 – Daily mean PM2.5** in both cities against the WHO 24-hour guideline (15 µg/m³).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for city, df in raw.items():
    d = daily_means(df['pm25'])          # NaN (gap in the line) on days with < 18 measured hours
    ax.plot(d.index, d, label=city, color=CITY_COLORS[city], lw=0.9)
ax.axhline(WHO_24H, color='k', ls='--', lw=1, label='WHO 24h guideline (15 µg/m³)')
ax.set(title='Daily mean PM2.5', ylabel='PM2.5 (µg/m³)')
ax.legend()
plt.show()

**Figure 2 – Distribution of hourly PM2.5** and **Figure 3 – Monthly boxplots** (seasonality).

In [ ]:
plot_df = pd.concat([df[['pm25']].assign(city=city) for city, df in raw.items()])
plot_df['month'] = plot_df.index.month
plot_df['hour'] = plot_df.index.hour
plot_df = plot_df.reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 4))
sns.histplot(data=plot_df, x='pm25', hue='city', bins=100, element='step', stat='density',
             common_norm=False, palette=CITY_COLORS, ax=ax)
ax.set(title='Distribution of hourly PM2.5', xlabel='PM2.5 (µg/m³)', xlim=(0, 250))
plt.show()

fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(data=plot_df, x='month', y='pm25', hue='city', palette=CITY_COLORS, showfliers=False, ax=ax)
ax.set(title='PM2.5 by month (box = median and quartiles; outliers hidden)', xlabel='Month', ylabel='PM2.5 (µg/m³)')
plt.show()

monthly = plot_df.groupby(['city', 'month'])['pm25'].agg(['mean', 'median']).round(1).unstack('city')
print('Hourly PM2.5 by calendar month (mean and median over all years, µg/m³):')
display(monthly)

**Figure 4 – Daily cycle:** mean PM2.5 by hour of day, split into the dry/cool half-year (Nov–Apr) and the wet/hot half-year (May–Oct).
Only **days with at least 18 measured hours** are used, so that every hour of the day is averaged over (almost) the same set of days.
With a plain mean by hour, an hour that is missing more often than others would be averaged over *different days*, and the shape of
the curve could reflect *which days have data* instead of the real daily cycle. With the chosen stations the coverage of every hour
of the day is close to the overall coverage (`worst_hour_cov` in Section 1.2), so this restriction changes little; it is kept as a safeguard.

In [ ]:
def complete_days_only(s, min_hours=MIN_HOURS_PER_DAY):
    """Keep only the hours of days that have at least `min_hours` measured hours."""
    measured_per_day = s.groupby(s.index.floor('D')).transform('count')
    return s[measured_per_day >= min_hours]


cycle_df = pd.concat([complete_days_only(df['pm25']).to_frame().assign(city=city) for city, df in raw.items()])
cycle_df['hour'] = cycle_df.index.hour
cycle_df['half_year'] = np.where(cycle_df.index.month.isin([11, 12, 1, 2, 3, 4]), 'Nov–Apr', 'May–Oct')

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, city in zip(axes, CITY_FILES):
    g = cycle_df[cycle_df.city == city]
    sns.lineplot(data=g, x='hour', y='pm25', hue='half_year', errorbar=None, ax=ax)
    ax.set(title=city, xlabel='Hour of day', ylabel='Mean PM2.5 (µg/m³)', xticks=range(0, 24, 3))
plt.tight_layout()
plt.show()

for city in CITY_FILES:
    for half, g in cycle_df[cycle_df.city == city].groupby('half_year'):
        by_hour = g.groupby('hour')['pm25'].mean()
        print(f'{city} {half} ({g.index.floor("D").nunique()} complete days): highest mean at {by_hour.idxmax()}h '
              f'({by_hour.max():.1f} µg/m³), lowest at {by_hour.idxmin()}h ({by_hour.min():.1f} µg/m³)')

**Figure 5 – Spearman correlation matrix** between PM2.5 and the weather variables, per city.

In [ ]:
num_cols = ['pm25', 'temperature_2m', 'relative_humidity_2m', 'precipitation',
            'wind_speed_10m', 'surface_pressure', 'cloud_cover']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (city, df) in zip(axes, raw.items()):
    sns.heatmap(df[num_cols].corr(method='spearman'), annot=True, fmt='.2f', cmap='RdBu_r',
                vmin=-1, vmax=1, ax=ax, cbar=False)
    ax.set_title(f'Spearman correlation – {city}')
    print(f'{city} – Spearman correlation of PM2.5 with:',
          df[num_cols].corr(method='spearman')['pm25'].drop('pm25').round(2).to_dict())
plt.tight_layout()
plt.show()

**Autocorrelation.** How much does the current value tell us about the value $k$ hours later? The lag-$k$ autocorrelation is the
Pearson correlation between the pairs $(x_t, x_{t-k})$ for all $t$ where both hours were measured
(`Series.autocorr(k)` in pandas). Lag 24h is the relevant one for a 24-hour-ahead forecast.

In [ ]:
ACF_LAGS = [1, 2, 3, 6, 12, 24, 48]
acf_table = pd.DataFrame({city: [df['pm25'].autocorr(lag) for lag in ACF_LAGS] for city, df in raw.items()},
                         index=pd.Index(ACF_LAGS, name='lag (hours)'))
display(acf_table.round(2))

**EDA findings:**


(Every number below is printed by a cell above: the statistics table in 1.3, the monthly table under Figure 3, the printout under Figure 4,
the Spearman printout under Figure 5, the autocorrelation table, and the missing-value tables of Section 2.1.)
- **Similar average pollution, different shapes:** mean hourly PM2.5 is 32.3 µg/m³ in Hanoi and 31.2 µg/m³ in HCMC.
  Hanoi is more volatile (std 20.4 vs 15.6 µg/m³).
- **WHO guideline (24-hour mean ≤ 15 µg/m³):** it is exceeded on **92.2% of the 476 valid days in Hanoi** and **97.6% of the 498 valid days in HCMC**
  (valid day = at least 18 measured hours). Both cities are almost always above the guideline.
- **Seasonality (monthly means):** Hanoi has a clear winter pollution season (December 54.1, January 48.7 µg/m³) and is cleanest in August (22.8),
  a factor of about 2.4. HCMC is also most polluted at the turn of the year (January 45.0, December 44.4) and cleanest in July (23.4).
  Figure 3 shows medians, which are a few µg/m³ lower (e.g. Hanoi December: median 51.7) because the distribution is right-skewed.
  The data covers **one** winter only, so each winter month is a single observation of that month.
- **Daily cycle (complete days only):** HCMC peaks at the 7h morning rush hour in both half-years (45.6 µg/m³ in Nov–Apr, 36.1 in May–Oct).
  Hanoi in Nov–Apr is highest at night (23h: 43.6) and lowest in the afternoon (15h: 37.9), consistent with a shallow night-time boundary layer
  trapping particles near the ground and daytime mixing diluting them; in May–Oct it peaks in the morning (9h: 35.3) and is lowest at 18h (18.5).
- **Correlation with weather (Spearman):** in Hanoi PM2.5 is lower with cloud cover (−0.29) and rain (−0.27) and higher with surface pressure (+0.26):
  dry, high-pressure winter air lets particles accumulate. In HCMC the strongest link is wind speed (−0.38): wind disperses particles.
  These are monotonic associations, not causal effects (the season drives several of them at once).
- **Autocorrelation:** 0.95 (Hanoi) / 0.88 (HCMC) at lag 1h, but only 0.59 / 0.47 at lag 24h. Forecasting 24h ahead is therefore much harder
  than forecasting the next hour. In HCMC the autocorrelation at 24h is higher than at 12h (0.47 vs 0.27): the same hour of the day tends to repeat.
- **Data quality:** Hanoi is missing 3.6% of PM2.5 hours (65 gaps, 48 of them ≤ 6h, longest 86h); the worst month is June 2026 (28% missing).
  HCMC is missing 0.6% (longest gap 8h). No value is ≤ 0 or > 1000 and no stuck-sensor run is found in either city (Section 2.2).

## 2. DATA PREPARATION

The steps below are written as **reusable functions** and applied **identically** to each city,
but every city is processed **independently**.

### 2.1. Missing value statistics

In [ ]:
def gap_lengths(series):
    """Length (hours) of each run of consecutive missing values."""
    is_na = series.isna()
    groups = (is_na != is_na.shift()).cumsum()
    return is_na.groupby(groups).sum()[lambda s: s > 0]


def missing_report(raw_dict):
    rows = []
    for city, df in raw_dict.items():
        gaps = gap_lengths(df['pm25'])
        rows.append({'City': city, 'Missing hours': int(df['pm25'].isna().sum()),
                     'Missing rate': f"{df['pm25'].isna().mean():.1%}",
                     'Gaps': len(gaps), 'Gaps ≤ 6h': int((gaps <= 6).sum()),
                     'Longest gap (hours)': int(gaps.max()) if len(gaps) else 0})
    return pd.DataFrame(rows)


display(missing_report(raw))
display((pd.concat({city: df.isna().mean() for city, df in raw.items()}, axis=1) * 100)
        .round(2).rename(columns=lambda c: f'{c} (% missing)'))

fig, ax = plt.subplots(figsize=(14, 4))
for city, df in raw.items():
    avail = df['pm25'].notna().resample('MS').mean() * 100
    ax.plot(avail.index, avail, marker='o', label=city, color=CITY_COLORS[city])
ax.set(title='Share of hours with PM2.5 data, by month', ylabel='% of hours with data', ylim=(0, 105))
ax.legend()
plt.show()

missing_by_month = pd.DataFrame({city: df['pm25'].isna().resample('MS').mean() * 100 for city, df in raw.items()})
missing_by_month.index = missing_by_month.index.strftime('%Y-%m')
print('% of hours with missing PM2.5, by month:')
display(missing_by_month.round(0).astype(int).T)

### 2.2. Cleaning invalid values and handling missing data

- **Invalid values:** PM2.5 ≤ 0 or > 1000 µg/m³ are sensor errors → set to `NaN`
  (a reference monitor reporting exactly 0.0 is a dropout, not clean air).
- **"Stuck" sensor:** the same value repeated for ≥ 6 consecutive hours (with PM2.5 > 0) indicates a frozen device → set to `NaN`.
- **No look-ahead when filling.** Time interpolation (`interpolate(method='time')`) fills a gap using the value measured **after**
  the gap, which is not known yet at forecast time $t$. If the features were built from interpolated values, the model would
  see up to 6 hours of the future (Section 2.4 prints how many samples this would concern). We therefore keep two versions:
  - `pm25_raw` — the measured values only. **The target is always a measured value** (never filled), so the evaluation is done on real data.
  - `pm25` — `pm25_raw` **forward-filled for at most 6 hours** (`ffill(limit=6)`): "the last known value, if it is at most 6h old".
    This is causal (uses only data ≤ $t$) and is used to build the features.
- **Long gaps (> 6 hours):** not filled (that would fabricate data) → samples whose current value or target is missing are dropped;
  remaining missing features are handled by `SimpleImputer` (median, fitted on the training set) inside the pipeline.

In [ ]:
MAX_FILL_GAP = 6
STUCK_HOURS = 6


def clean_city(df):
    df = df.copy()
    pm = df['pm25']
    invalid = (pm <= 0) | (pm > 1000)
    run_id = (pm != pm.shift()).cumsum()
    run_len = pm.groupby(run_id).transform('size')
    stuck = (run_len >= STUCK_HOURS) & (pm > 0) & pm.notna()
    df.loc[invalid | stuck, 'pm25'] = np.nan
    df['pm25_raw'] = df['pm25']                                # measured values only (used for the target)
    df['pm25'] = df['pm25_raw'].ffill(limit=MAX_FILL_GAP)      # causal fill (used for the features)
    return df, int(invalid.sum()), int(stuck.sum())


clean = {}
for city, df in raw.items():
    clean[city], n_invalid, n_stuck = clean_city(df)
    before, after = df['pm25'].isna().mean(), clean[city]['pm25'].isna().mean()
    print(f'{city}: {n_invalid} values ≤ 0 or > 1000, {n_stuck} stuck-sensor hours | '
          f'missing PM2.5: {before:.1%} → {after:.1%} after cleaning + causal forward-fill (features only)')


def first_short_gap(series, min_len=3, max_len=MAX_FILL_GAP):
    is_na = series.isna()
    groups = (is_na != is_na.shift()).cumsum()
    sizes = is_na.groupby(groups).transform('sum')
    mask = is_na & (sizes >= min_len) & (sizes <= max_len)
    return series.index[mask][0] if mask.any() else None


for city in clean:
    gap_time = first_short_gap(clean[city]['pm25_raw'])
    if gap_time is not None:
        break
if gap_time is not None:
    window = clean[city].loc[gap_time - pd.Timedelta('36h'): gap_time + pd.Timedelta('36h')]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.step(window.index, window['pm25'], where='post', ls='--', color='orange', label='Forward-filled (feature input)')
    ax.plot(window.index, window['pm25_raw'], 'o-', color=CITY_COLORS[city], label='Measured (pm25_raw, target)')
    ax.set(title=f'Causal forward-fill of a short gap – {city}', ylabel='PM2.5 (µg/m³)')
    ax.legend()
    plt.show()

### 2.3. Outlier detection (IQR)
IQR rule: values outside $[Q1 - 3\cdot IQR,\ Q3 + 3\cdot IQR]$ are treated as extreme outliers.

In [ ]:
def iqr_bounds(s, k=3.0):
    q1, q3 = s.quantile([0.25, 0.75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)


fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (city, df) in zip(axes, clean.items()):
    lo, hi = iqr_bounds(df['pm25_raw'])
    out = df.loc[df['pm25_raw'] > hi, 'pm25_raw']
    top_days = out.resample('D').count().nlargest(3).index.strftime('%Y-%m-%d').tolist()
    print(f'{city}: upper bound = {hi:.1f} µg/m³ | {len(out)} outlier hours '
          f'({len(out) / df.pm25_raw.notna().sum():.2%}) | days with most outliers: {top_days}')
    ax.plot(df.index, df['pm25_raw'], lw=0.4, color=CITY_COLORS[city])
    ax.scatter(out.index, out, s=6, color='k', label='Outlier (IQR, k=3)')
    ax.axhline(hi, color='k', ls='--', lw=1)
    ax.set(title=city, ylabel='PM2.5 (µg/m³)')
    ax.legend()
plt.tight_layout()
plt.show()

**Outlier decision:** PM2.5 peaks are **real events** (winter pollution episodes, straw burning, fireworks...),
so they are **not removed** from the target (removing them would make the evaluation look better than it is). Instead, the PM2.5
**features** (lags, rolling statistics) go through a custom `IQRClipper` transformer (bounds learned **on the training set only**)
to reduce the influence of extreme values on the linear model.

### 2.4. Feature engineering

Each sample at time $t$ has target `y = pm25_raw(t + 24h)` (a **measured** value). The features are built from the forward-filled
series and use only information **available at $t$** (no future leakage):

| Group | Features | Type |
|---|---|---|
| PM2.5 history | `pm25_lag_0, 1, 2, 3, 6, 12, 23` (value at $t-k$), `pm25_roll_mean_6/24`, `pm25_roll_std_24`, `pm25_diff_24` | numerical |
| Weather at $t$ | temperature, humidity, precipitation, wind speed, pressure, cloud cover, `rain_24h` (total rain over the last 24h) | numerical |
| Calendar of the forecast time | `hour_sin/cos` (cyclic hour encoding) | numerical |
| | `season` (climate season of each city), `day_of_week`, `wind_sector` (8 wind directions) | **categorical** |

> `pm25_lag_0` is the value at the **same hour on the previous day** relative to the forecast time, so the
> **Persistence** baseline (`ŷ(t+24) = pm25(t)`) is also the 24h *seasonal naive* baseline.

In [ ]:
SEASONS = {
    'Hanoi': {12: 'Winter', 1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring', 5: 'Spring',
              6: 'Summer', 7: 'Summer', 8: 'Summer', 9: 'Autumn', 10: 'Autumn', 11: 'Autumn'},
    'HCMC': {m: ('Dry season' if m in (12, 1, 2, 3, 4) else 'Wet season') for m in range(1, 13)},
}
WIND_SECTORS = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
LAGS = [0, 1, 2, 3, 6, 12, 23]
WEATHER_RAW = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m',
               'surface_pressure', 'cloud_cover']
WEATHER_FEATURES = WEATHER_RAW + ['rain_24h']
PM_FEATURES = [f'pm25_lag_{k}' for k in LAGS] + ['pm25_roll_mean_6', 'pm25_roll_mean_24',
                                                 'pm25_roll_std_24', 'pm25_diff_24']
CYCLIC_FEATURES = ['hour_sin', 'hour_cos']
CAT_FEATURES = ['season', 'day_of_week', 'wind_sector']
FEATURES = PM_FEATURES + WEATHER_FEATURES + CYCLIC_FEATURES + CAT_FEATURES
TARGET = 'target'


def build_features(df, city, horizon=HORIZON):
    """Build the features at each time t (using only data ≤ t)."""
    pm = df['pm25']
    X = pd.DataFrame(index=df.index)
    for k in LAGS:
        X[f'pm25_lag_{k}'] = pm.shift(k)
    X['pm25_roll_mean_6'] = pm.rolling(6, min_periods=3).mean()
    X['pm25_roll_mean_24'] = pm.rolling(24, min_periods=12).mean()
    X['pm25_roll_std_24'] = pm.rolling(24, min_periods=12).std()
    X['pm25_diff_24'] = pm - pm.shift(24)
    X[WEATHER_RAW] = df[WEATHER_RAW]
    X['rain_24h'] = df['precipitation'].rolling(24, min_periods=1).sum()

    target_time = df.index + pd.Timedelta(hours=horizon)
    X['hour_sin'] = np.sin(2 * np.pi * target_time.hour / 24)
    X['hour_cos'] = np.cos(2 * np.pi * target_time.hour / 24)
    X['season'] = pd.Series(target_time.month, index=df.index).map(SEASONS[city])
    X['day_of_week'] = target_time.day_name()
    sector_idx = ((df['wind_direction_10m'] + 22.5) % 360) // 45
    X['wind_sector'] = sector_idx.map(lambda i: WIND_SECTORS[int(i)] if pd.notna(i) else np.nan)
    return X


def make_dataset(df, city, horizon=HORIZON):
    """Add target = measured pm25_raw(t + horizon); keep only samples with a target and a current value."""
    X = build_features(df, city, horizon)
    X[TARGET] = df['pm25_raw'].shift(-horizon)
    return X.dropna(subset=[TARGET, 'pm25_lag_0'])


datasets = {city: make_dataset(df, city) for city, df in clean.items()}
for city, d in datasets.items():
    filled_now = clean[city]['pm25_raw'].reindex(d.index).isna().mean()   # current value comes from the forward-fill
    print(f'{city}: {len(d):,} samples | {len(FEATURES)} features '
          f'({len(FEATURES) - len(CAT_FEATURES)} numerical + {len(CAT_FEATURES)} categorical) | '
          f'{filled_now:.1%} of samples have a forward-filled current value '
          f'(with two-sided interpolation these would have used future measurements)')
datasets['Hanoi'].head()

### 2.5. Chronological split: Development / Test, and walk-forward folds
A time series **must not be shuffled** (that would leak the future into the past), so every split follows **time order** and uses
the **same calendar dates for both cities**, so the two cities are compared over the same weather and season.

- **Test** = samples issued from **01/09/2026** to the end of the data (the last 2.5 months). It is used **once**, at the very end (Section 4).
- **Development** = all samples issued **before** 01/09/2026 − 24h. The extra `HORIZON` = 24 hours makes sure that the **target**
  ($t + 24h$) of every Development sample is measured before the first Test forecast is issued.

The data covers only one winter, and a single fixed Validation month (or 15% block) would fall in one season: a model would then be
chosen on, say, clean summer hours and judged on the polluted season. Model selection therefore uses **walk-forward validation**
(also called evaluation on a rolling forecasting origin, Tashman 2000; Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, §5.10):
for every month $m$ from **10/2025 to 08/2026** (11 folds), with $s_m$ the first hour of month $m$,

$$\text{Train}_m = \{t \in \text{Dev} : t < s_m - 24h\}, \qquad \text{Val}_m = \{t \in \text{Dev} : s_m \le t < s_{m+1}\}.$$

Each model is refitted on $\text{Train}_m$ (an **expanding** window: everything known before the month) and forecasts every sample
of $\text{Val}_m$. The 11 validation months do not overlap, so each Development hour from 10/2025 onwards is forecast **exactly once**,
by a model that never saw it, and the validation set covers the whole winter 2025–26. The first three months (07–09/2025) are only
used for training, so the first fold has a minimum amount of history.

In [ ]:
TEST_START = pd.Timestamp('2026-09-01')
WALK_FORWARD_MONTHS = pd.period_range('2025-10', '2026-08', freq='M')


def dev_test_split(d, test_start=TEST_START, gap_hours=HORIZON):
    """Development = samples whose target is measured before test_start; Test = samples issued from test_start."""
    dev = d[d.index < test_start - pd.Timedelta(hours=gap_hours)]
    test = d[d.index >= test_start]
    return dev, test


def walk_forward_folds(dev, months=WALK_FORWARD_MONTHS, gap_hours=HORIZON):
    """For each month m: (m, train = samples whose target is known before m starts, val = samples issued in m)."""
    folds = []
    for m in months:
        start, end = m.start_time, (m + 1).start_time
        train = dev[dev.index < start - pd.Timedelta(hours=gap_hours)]
        val = dev[(dev.index >= start) & (dev.index < end)]
        folds.append((m, train, val))
    return folds


splits = {city: dev_test_split(d) for city, d in datasets.items()}

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, (city, (dev, te)) in zip(axes, splits.items()):
    pre_wf = dev[dev.index < WALK_FORWARD_MONTHS[0].start_time]
    wf = dev[dev.index >= WALK_FORWARD_MONTHS[0].start_time]
    for part, name, color in [(pre_wf, 'Development – training only', 'tab:blue'),
                              (wf, 'Development – walk-forward validation months', 'tab:orange'),
                              (te, 'Test', 'tab:green')]:
        daily = part[TARGET].resample('D').mean()
        ax.plot(daily.index, daily, color=color, lw=0.8,
                label=f'{name}: {len(part):,} samples ({part.index.min():%d/%m/%Y} – {part.index.max():%d/%m/%Y})')
    for m in WALK_FORWARD_MONTHS:
        ax.axvline(m.start_time, color='gray', lw=0.5, ls=':')
    ax.set(title=f'{city} (dotted lines: start of each walk-forward fold)', ylabel='Daily mean target PM2.5 (µg/m³)')
    ax.legend(loc='upper left', fontsize=9)
    print(f'{city} split: Development {len(dev):,} ({dev.index.min():%Y-%m-%d} → {dev.index.max():%Y-%m-%d %H:%M}) | '
          f'Test {len(te):,} ({te.index.min():%Y-%m-%d} → {te.index.max():%Y-%m-%d %H:%M}) samples | '
          f'mean target: Development {dev[TARGET].mean():.1f}, Test {te[TARGET].mean():.1f} µg/m³')
plt.tight_layout()
plt.show()

rows = []
for city, (dev, te) in splits.items():
    for m, tr, va in walk_forward_folds(dev):
        rows.append({'city': city, 'month': str(m), 'train': len(tr), 'validation': len(va)})
fold_sizes = pd.DataFrame(rows).pivot(index='month', columns='city', values=['train', 'validation'])
print('Walk-forward folds: number of samples in the training window and in the validation month')
display(fold_sizes)

**Figure caption (split):** both cities use the same dates. Development has 9,598 (Hanoi) / 10,076 (HCMC) samples and Test 1,775 samples in each city
(01/09/2026 – 13/11/2026). The mean target is similar in Development and Test (Hanoi 32.3 vs 32.7, HCMC 31.0 vs 32.4 µg/m³), but the Test period
is the **start** of the polluted season, not its peak. The walk-forward training window grows from 2,075 (Hanoi) / 2,100 (HCMC) samples in fold 10/2025 to 8,870 / 9,336 in fold 08/2026;
each validation month has 653–743 samples, except Hanoi in June 2026 (487) because of the gaps in that month.

### 2.6. Preprocessing pipeline (imputation, outlier clipping, encoding, scaling)

| Column group | Steps |
|---|---|
| PM2.5 lags/rolling | `SimpleImputer(median)` → `IQRClipper(k=3)` → `StandardScaler` |
| Weather + hour cycle | `SimpleImputer(median)` → `StandardScaler` |
| Categorical | `SimpleImputer(most_frequent)` → `OneHotEncoder` |

Everything lives inside a `ColumnTransformer` + `Pipeline`, so all parameters (medians, IQR bounds, mean/std, categories)
are **learned on the data the pipeline is fitted on** (the training window of each fold, or the whole Development set
for the final model), never on the data it is evaluated on → no data leakage.

In [ ]:
class IQRClipper(BaseEstimator, TransformerMixin):
    """Clip (winsorize) each column to [Q1 - k·IQR, Q3 + k·IQR], bounds learned on the fitted data."""

    def __init__(self, k=3.0):
        self.k = k

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1, q3 = np.nanpercentile(X, [25, 75], axis=0)
        self.lower_, self.upper_ = q1 - self.k * (q3 - q1), q3 + self.k * (q3 - q1)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


def build_preprocessor():
    return ColumnTransformer([
        ('pm', Pipeline([('impute', SimpleImputer(strategy='median')),
                         ('clip', IQRClipper(k=3.0)),
                         ('scale', StandardScaler())]), PM_FEATURES),
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), WEATHER_FEATURES + CYCLIC_FEATURES),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                          ('onehot', OneHotEncoder(handle_unknown='ignore'))]), CAT_FEATURES),
    ], sparse_threshold=0)


tr = splits['Hanoi'][0]   # Development set of Hanoi
pre = build_preprocessor().fit(tr[FEATURES])
Xt = pd.DataFrame(pre.transform(tr[FEATURES]), columns=pre.get_feature_names_out(), index=tr.index)
print(f'After preprocessing: {Xt.shape[1]} columns (from {len(FEATURES)} original columns, one-hot expands the categoricals)')
print('Missing values left after the pipeline:', int(Xt.isna().sum().sum()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(tr['pm25_lag_0'], bins=80, ax=axes[0], color='gray')
axes[0].set(title='pm25_lag_0 – before preprocessing (Hanoi, Development)', xlabel='µg/m³')
sns.histplot(Xt['pm__pm25_lag_0'], bins=80, ax=axes[1], color='teal')
axes[1].set(title='After IQRClipper + StandardScaler', xlabel='z-score')
plt.tight_layout()
plt.show()

## 3. ANALYSIS AND MODELING

### 3.1. Models compared
| Model | Role | Why |
|---|---|---|
| **Persistence (baseline)** | `ŷ(t+24) = pm25(t)` | Standard time series baseline (= 24h seasonal naive) |
| **Mean 24h (baseline)** | `ŷ(t+24) = mean(pm25[t-23 … t])` | Smooths out the noise of a single hour |
| **Naive blend (baseline)** | `ŷ = w·pm25(t) + (1−w)·mean_24h(t)` | The **strongest naive baseline**: `w` is learned on the training data (grid 0, 0.05, …, 1, minimizing MAE). An ML model is only really useful if it beats **this** one, not just Persistence |
| **Ridge Regression** | Regularized linear model | Simple and interpretable; tests how linear the relationship is |
| **Random Forest** | Tree ensemble (bagging) | Captures non-linear effects and weather × hour interactions |
| **HistGradientBoosting** | Tree ensemble (boosting) | Strong and fast on large tabular data |

**Hyperparameter tuning:** `GridSearchCV` with `TimeSeriesSplit(n_splits=3, gap=24)` on the **Development** set, scored by `MAE`.
The validation fold always comes **after** the training fold, and the `gap` drops 24 samples between them so that the targets
(24h ahead) of the last training samples do not fall inside the validation fold. All three ML models are tuned
(Random Forest with a small grid to keep the run time reasonable on Colab). The printout flags any selected value that lies at
the smallest or largest end of its grid, because the best value could then lie outside the grid, and shows the best and worst
cross-validated MAE over the grid, i.e. how much the choice of hyperparameters matters at all for each model. (For Ridge, the smallest `alpha` is not a concern: as `alpha` → 0 Ridge tends to ordinary least squares.) The hyperparameters are then **fixed** for every
walk-forward fold and for the final model.

**Model selection:** walk-forward validation (Section 2.5). Every model, baselines included, is refitted on each training window
$\text{Train}_m$ (so the blend weight $w$ is also re-learned in every fold) and forecasts month $m$. The scores are computed on the
**pooled** forecasts of the 11 months, e.g. $\mathrm{MAE} = \sum_m \sum_{i \in \text{Val}_m} |y_i - \hat y_i| \, / \, \sum_m |\text{Val}_m|$,
and the ML model with the lowest pooled MAE is the **best model** of the city.

*Stated limitation:* the hyperparameters were tuned on the whole Development set, which contains the walk-forward months, so the
walk-forward errors of the three ML models are **slightly optimistic** (the baselines have no hyperparameters). Re-tuning inside each
of the 11 folds (nested tuning) would remove this bias but multiplies the run time by about 11. The Test set (Section 4) is not
affected: it was never used for tuning or selection.

The `CityForecaster` class wraps the whole workflow for **one** city so that it can be repeated for both.

In [ ]:
class PersistenceBaseline(BaseEstimator, RegressorMixin):
    """Forecast PM2.5 HORIZON hours ahead with the current value."""

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        return np.asarray(X['pm25_lag_0'], dtype=float)


class MeanBaseline(BaseEstimator, RegressorMixin):
    """Forecast PM2.5 HORIZON hours ahead with the mean of the last 24 hours."""

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        return np.asarray(X['pm25_roll_mean_24'].fillna(X['pm25_lag_0']), dtype=float)


class NaiveBlend(BaseEstimator, RegressorMixin):
    """w·(current value) + (1 − w)·(24h mean); w is chosen on the fitted data to minimize MAE."""

    def __init__(self, weights=tuple(np.round(np.linspace(0, 1, 21), 2))):
        self.weights = weights

    @staticmethod
    def _parts(X):
        now = np.asarray(X['pm25_lag_0'], dtype=float)
        return now, np.asarray(X['pm25_roll_mean_24'].fillna(X['pm25_lag_0']), dtype=float)

    def fit(self, X, y):
        now, mean24 = self._parts(X)
        maes = [np.mean(np.abs(np.asarray(y) - (w * now + (1 - w) * mean24))) for w in self.weights]
        self.w_ = float(self.weights[int(np.argmin(maes))])
        return self

    def predict(self, X):
        now, mean24 = self._parts(X)
        return self.w_ * now + (1 - self.w_) * mean24


BASELINES = ['Persistence (baseline)', 'Mean 24h (baseline)', 'Naive blend (baseline)']


def regression_metrics(y_true, y_pred):
    return {'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2': r2_score(y_true, y_pred)}


def make_models():
    return {
        'Persistence (baseline)': PersistenceBaseline(),
        'Mean 24h (baseline)': MeanBaseline(),
        'Naive blend (baseline)': NaiveBlend(),
        'Ridge': Pipeline([('pre', build_preprocessor()), ('model', Ridge())]),
        'RandomForest': Pipeline([('pre', build_preprocessor()), ('model', RandomForestRegressor(
            n_estimators=200, max_depth=12, min_samples_leaf=10, max_features=0.5,
            n_jobs=-1, random_state=RANDOM_STATE))]),
        # early_stopping=False: the default 'auto' switches early stopping on above 10,000 samples and then holds out a
        # *random* 10% of the rows (not a time-ordered split), so the model would behave differently in the CV folds
        # (< 10,000 rows) and in the final fit; with False the number of trees is always exactly max_iter.
        'HistGradientBoosting': Pipeline([('pre', build_preprocessor()),
                                          ('model', HistGradientBoostingRegressor(early_stopping=False,
                                                                                  random_state=RANDOM_STATE))]),
    }


PARAM_GRIDS = {
    'Ridge': {'model__alpha': [0.01, 0.1, 1, 10, 100, 1000]},
    'RandomForest': {'model__max_depth': [4, 6, 8, 12],
                     'model__min_samples_leaf': [5, 10, 30, 100]},
    'HistGradientBoosting': {'model__learning_rate': [0.01, 0.03, 0.1],
                             'model__max_iter': [100, 200, 400],
                             'model__max_leaf_nodes': [4, 7, 15],
                             'model__l2_regularization': [1.0, 10.0, 100.0]},
}


class CityForecaster:
    def __init__(self, city, dev, test):
        self.city, self.dev, self.test = city, dev, test
        self.models = make_models()
        self.best_params, self.cv_mae_range = {}, {}

    def tune(self):
        for name, grid in PARAM_GRIDS.items():
            search = GridSearchCV(self.models[name], grid, cv=TimeSeriesSplit(n_splits=3, gap=HORIZON),
                                  scoring='neg_mean_absolute_error', n_jobs=-1)
            search.fit(self.dev[FEATURES], self.dev[TARGET])
            self.models[name] = search.best_estimator_
            self.best_params[name] = search.best_params_
            self.cv_mae_range[name] = (-search.cv_results_['mean_test_score'].max(),
                                       -search.cv_results_['mean_test_score'].min())
        return self

    def walk_forward(self):
        """Refit every model on each expanding training window and forecast the next month (Section 2.5)."""
        parts = []
        for month, train, val in walk_forward_folds(self.dev):
            p = pd.DataFrame({'actual': val[TARGET], 'fold': str(month)})
            for name, model in self.models.items():
                m = clone(model)                      # same hyperparameters, fitted from scratch
                m.fit(train[FEATURES], train[TARGET])
                p[name] = m.predict(val[FEATURES])
            parts.append(p)
        self.wf_predictions = pd.concat(parts)
        return pd.DataFrame([{'city': self.city, 'model': name,
                              **regression_metrics(self.wf_predictions['actual'], self.wf_predictions[name])}
                             for name in self.models])

    def evaluate_on_test(self):
        """Refit on the whole Development set, then evaluate once on Test."""
        rows, self.test_predictions = [], pd.DataFrame({'actual': self.test[TARGET]})
        for name, m in self.models.items():
            m.fit(self.dev[FEATURES], self.dev[TARGET])
            pred = m.predict(self.test[FEATURES])
            self.test_predictions[name] = pred
            rows.append({'city': self.city, 'model': name, **regression_metrics(self.test[TARGET], pred)})
        return pd.DataFrame(rows)


forecasters = {city: CityForecaster(city, *parts).tune() for city, parts in splits.items()}
for city, f in forecasters.items():
    print(f'{city} – best hyperparameters:')
    for name, params in f.best_params.items():
        # a value at the smallest or largest end of its grid means the optimum may lie outside the grid
        at_edge = [k.replace('model__', '') for k, v in params.items()
                   if len(PARAM_GRIDS[name][k]) > 1 and v in (min(PARAM_GRIDS[name][k]), max(PARAM_GRIDS[name][k]))]
        best_cv, worst_cv = f.cv_mae_range[name]
        print(f'   {name}: {params} | at the edge of the grid: {at_edge if at_edge else "none"} | '
              f'CV MAE best {best_cv:.2f}, worst {worst_cv:.2f} over the grid')
    blend = clone(f.models['Naive blend (baseline)']).fit(f.dev[FEATURES], f.dev[TARGET])
    print(f'   Naive blend: w = {blend.w_:.2f} (weight of the current value, learned on the whole Development set)')

### 3.2. Walk-forward validation (model selection)
Pooled scores over the 11 monthly folds (10/2025 – 08/2026), then the MAE of each model in each month.

In [ ]:
wf_results = pd.concat([f.walk_forward() for f in forecasters.values()], ignore_index=True)
wf_table = wf_results.pivot(index='model', columns='city', values=['MAE', 'RMSE', 'R2'])
display(wf_table.reindex(columns=list(CITY_FILES), level=1).reindex(list(make_models())).round(3))

ml_wf = wf_results[~wf_results.model.isin(BASELINES)]
best_model = ml_wf.loc[ml_wf.groupby('city')['MAE'].idxmin()].set_index('city')['model']
print('Best model (lowest pooled walk-forward MAE among the ML models):', best_model.to_dict())

for city, f in forecasters.items():
    w_per_fold = {}
    for month, train, val in walk_forward_folds(f.dev):
        w_per_fold[str(month)] = clone(f.models['Naive blend (baseline)']).fit(train[FEATURES], train[TARGET]).w_
    print(f'{city} – Naive blend weight w learned in each fold:', w_per_fold)

**Figure – Walk-forward MAE by month** for every model. Each point is the MAE of the forecasts issued in that month by a model trained
only on data known before the month started. The bars show the mean measured PM2.5 (target) of the month, for context.
The bias tables below give the **bias** of the best model and of the Naive blend in each month, $\frac{1}{|\text{Val}_m|}\sum_{i \in \text{Val}_m} (\hat y_i - y_i)$:
a negative bias means the forecasts are too low on average.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
monthly_mae = {}
for ax, (city, f) in zip(axes, forecasters.items()):
    p = f.wf_predictions
    abs_err = p[list(f.models)].sub(p['actual'], axis=0).abs()
    monthly_mae[city] = abs_err.groupby(p['fold']).mean()
    level = p.groupby('fold')['actual'].mean()
    ax2 = ax.twinx()
    ax2.bar(range(len(level)), level, color='lightgray', alpha=0.6, width=0.7)
    ax2.set_ylabel('Mean measured PM2.5 (µg/m³)', color='gray')
    ax2.grid(False)
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)
    for name in f.models:
        style = dict(ls='--', lw=1.2) if name in BASELINES else dict(lw=2, marker='o', ms=4)
        ax.plot(range(len(level)), monthly_mae[city][name].reindex(level.index), label=name, **style)
    ax.set_xticks(range(len(level)), level.index, rotation=45)
    ax.set(title=f'{city} – walk-forward MAE by month', ylabel='MAE (µg/m³)')
axes[0].legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

for city in CITY_FILES:
    print(f'{city} – walk-forward MAE by month (µg/m³):')
    display(monthly_mae[city].T.reindex(list(make_models())).round(2))

# Bias = mean(forecast − actual) per month: negative = the model forecasts too low on average
for city, f in forecasters.items():
    p = f.wf_predictions
    cols = [best_model[city], 'Naive blend (baseline)']
    bias = p[cols].sub(p['actual'], axis=0).groupby(p['fold']).mean()
    bias['mean measured PM2.5'] = p.groupby('fold')['actual'].mean()
    print(f'{city} – walk-forward bias by month (mean of forecast − actual, µg/m³):')
    display(bias.T.round(1))

**Walk-forward findings** (Section 3.2; pooled over the 11 months 10/2025 – 08/2026):
- **Hyperparameters:** cross-validation prefers simple models: HGB with at most 4 leaves per tree in both cities, Random Forest with depth 8 (Hanoi) or 4 (HCMC).
  Several selected values lie at the edge of the grid (flagged in the printout). The grid was widened twice and the selected values moved to the
  simpler end each time; we stopped there and state it as a limitation. A noisy 24h-ahead target with one year of data favours simple models.
- **Naive blend:** `w` stays between 0.65 and 0.75 in Hanoi and 0.45–0.55 in HCMC across the folds, so the blend is a stable reference.
  It clearly beats Persistence (MAE 13.358 vs 13.788 in Hanoi, 9.921 vs 10.895 in HCMC).
- **ML vs the Naive blend:** all three ML models beat it on the pooled forecasts. **Hanoi:** RandomForest 12.453, Ridge 12.521, HGB 12.646 vs 13.358.
  **HCMC:** HGB 9.593, Ridge 9.630, RandomForest 9.751 vs 9.921. The selected models are **RandomForest (Hanoi)** and **HistGradientBoosting (HCMC)**,
  but the pooled MAEs of the three ML models differ by less than 2%.
- **The gain is not uniform over the year (figure above).** In Hanoi, RandomForest is far better than the blend in the winter peak
  (January 15.54 vs 19.36, February 16.31 vs 20.53) but **worse in November 2025 (17.54 vs 14.96) and April 2026 (12.12 vs 11.26)**.
  In HCMC, HGB is better than the blend in the polluted months October–January (e.g. December 12.87 vs 14.67) and in May and August,
  but worse in February (11.45 vs 10.40), March, April, June and July.
- **Why (bias table):** the ML models lag behind changes of the pollution level. When the level rises, they forecast too low:
  Hanoi RandomForest has a bias of −7.8 µg/m³ in November (mean measured PM2.5 38.4) and −7.1 in April (40.1, up from 30.3 in March);
  HCMC HGB −6.9 in October. When it falls they forecast too high (HCMC February +5.8, as the mean drops from 44.8 to 32.9).
  The Naive blend has no intercept and simply follows the current level, so its monthly bias stays within ±2.5 µg/m³.
  In November 2025 the Hanoi model had only seen July–October, the cleaner months.

## 4. SUMMARY AND EXTENSIONS

### 4.1. Results on the independent Test set
Each model is refitted on the whole Development set (with its hyperparameters fixed) and evaluated **once** on Test
(01/09/2026 to the end of the data). The **best model of each city was chosen by walk-forward validation** (Section 3.2), not on Test,
so the Test set is used only for reporting and its scores are not biased by model selection.
*Note:* the Test period is the **start** of the polluted season (September–November), not its peak (December–February);
the winter is covered by the walk-forward months.

In [ ]:
test_results = pd.concat([f.evaluate_on_test() for f in forecasters.values()], ignore_index=True)
for ref, col in [('Persistence (baseline)', 'MAE vs Persistence'), ('Naive blend (baseline)', 'MAE vs Naive blend')]:
    ref_mae = test_results[test_results.model == ref].set_index('city')['MAE']
    test_results[col] = test_results.apply(lambda r: 1 - r['MAE'] / ref_mae[r['city']], axis=1)
display(test_results.set_index(['city', 'model']).style.format(
    {'MAE': '{:.2f}', 'RMSE': '{:.2f}', 'R2': '{:.3f}', 'MAE vs Persistence': '{:+.1%}', 'MAE vs Naive blend': '{:+.1%}'}))
print('Test:', {c: f.test.index.min().strftime('%Y-%m-%d') + ' → ' + f.test.index.max().strftime('%Y-%m-%d')
                for c, f in forecasters.items()})
print('Best model (chosen by walk-forward validation, Section 3.2):', best_model.to_dict())

### 4.2. Question: *How much better than the baseline is machine learning in each city?*
**Figure 6 – Test MAE** of each model, per city. Bar labels: % MAE reduction vs. the **Naive blend** baseline
(the strongest naive forecast), so a negative label means "worse than the naive blend".

In [ ]:
MODEL_ORDER = list(make_models())
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=test_results, x='model', y='MAE', hue='city', order=MODEL_ORDER,
            hue_order=list(CITY_FILES), palette=CITY_COLORS, ax=ax)
for container, city in zip(ax.containers, CITY_FILES):
    sub = test_results[test_results.city == city].set_index('model')
    ax.bar_label(container, labels=[f"{sub.loc[m, 'MAE vs Naive blend']:+.1%}" for m in MODEL_ORDER],
                 fontsize=8, padding=2)
ax.set(title='Test MAE (lower is better). Labels: MAE reduction vs. Naive blend (+ = better)', xlabel='', ylabel='MAE (µg/m³)')
ax.tick_params(axis='x', labelrotation=15)
ax.set_ylim(0, test_results['MAE'].max() * 1.25)
ax.legend(title='', loc='upper center', ncol=2)
plt.show()

**Figure 6 caption:** On the independent Test set (01/09 – 13/11/2026), the best model of each city (chosen by walk-forward validation) reaches
**MAE 10.91 µg/m³ in Hanoi (RandomForest)** and **11.07 µg/m³ in HCMC (HistGradientBoosting)**. Against the two reference forecasts:

| City | vs Persistence | vs Naive blend (strongest naive baseline) |
|---|---|---|
| Hanoi (RandomForest) | 12.07 → 10.91 (−9.6%) | 11.53 → 10.91 (**−5.3%**) |
| HCMC (HistGradientBoosting) | 12.88 → 11.07 (−14.1%) | 11.93 → 11.07 (**−7.2%**) |

Comparing only with Persistence overstates the benefit of machine learning: part of the gain comes from simply smoothing the current value
with its 24h mean. The honest answer to the question is therefore **about 5% in Hanoi and 7% in HCMC** beyond the best naive forecast on Test
(6.8% / 3.3% on the walk-forward months). On Test, Ridge is actually the best model in Hanoi (10.55, −8.4% vs the blend), although it was not selected:
the ML models are too close for the selection to be reliable (Section 4.3). In HCMC the Test R² is low for every model (at most 0.166):
day-to-day variations in the autumn are hard to forecast from the current level and weather.

Only **measured** PM2.5 values are used as targets (no interpolated targets), and the features use a causal forward-fill only.
(Tree-model results can shift by a few hundredths of µg/m³ between scikit-learn versions; Ridge and the baselines do not change.)

### 4.3. Are the differences real? Block-bootstrap confidence intervals
Thousands of hourly forecasts sound like a lot, but forecast errors are **strongly autocorrelated** (an error at 10h is very
likely to be followed by a similar error at 11h), so the number of *independent* observations is much smaller, and an
ordinary i.i.d. bootstrap over hours would give intervals that are far too narrow.

**Method (non-overlapping block bootstrap, Carlstein 1986; the block idea for dependent data is also in Künsch 1989).**
For a comparison of model $A$ (the best model) with model $B$ on one evaluation set (the pooled walk-forward forecasts, or Test):
1. Split the samples into $k$ consecutive **3-day calendar blocks** of forecast time $t+24h$. Blocks do not overlap; because of
   missing hours they do not all contain the same number of samples. Let $S_j^A = \sum_{i \in j} |y_i - \hat y_i^A|$ be the sum of
   absolute errors of $A$ in block $j$ (same for $B$, on the **same** hours — the comparison is *paired*).
2. Draw $k$ blocks **with replacement**, i.e. counts $(c_1, \dots, c_k) \sim \mathrm{Multinomial}(k; 1/k, \dots, 1/k)$, and compute
   $g^* = 1 - \sum_j c_j S_j^A \, / \, \sum_j c_j S_j^B$ (the hour counts cancel because both models are scored on the same hours).
3. Repeat 2,000 times; the 2.5th and 97.5th percentiles of $g^*$ form the 95% **percentile** confidence interval of the gain
   $g = 1 - \mathrm{MAE}_A / \mathrm{MAE}_B$.

An interval that excludes 0 corresponds (approximately) to rejecting "no difference" in a two-sided test at the 5% level.
**Choice of block length:** blocks must be long enough that the error differences of different blocks are close to independent.
The cell below prints, on the walk-forward forecasts, the autocorrelation of the *daily mean* error difference between the best model and the
Naive blend at lags of 1, 2 and 3 days; the 3-day block length is checked against it, and 1-day and 7-day blocks are reported as a sensitivity check.

**Limitations of the method, stated explicitly:** (1) the percentile interval is an approximation, especially on Test, which has few blocks
(the number of blocks is the `blocks` column of the table); (2) we report 16 intervals without a multiple-comparison correction, so they are
**descriptive** — the conclusions are based on the four comparisons fixed in advance: best model vs Naive blend, on walk-forward and on Test,
for each city; (3) blocks assume the error process is roughly stationary within the evaluation period, which is only approximately true over
11 months with a winter in the middle; (4) on walk-forward, the best model was selected on these same forecasts and its hyperparameters
were tuned on the Development set, so its walk-forward gain is slightly optimistic — the Test interval is the unbiased one.

In [ ]:
BLOCK_DAYS = 3
N_BOOT = 2000

# Block-length check on the walk-forward forecasts: autocorrelation of the daily mean |error| difference (best model − Naive blend)
for city, f in forecasters.items():
    p = f.wf_predictions
    diff = (p[best_model[city]] - p['actual']).abs() - (p['Naive blend (baseline)'] - p['actual']).abs()
    daily_diff = diff.groupby((diff.index + pd.Timedelta(hours=HORIZON)).floor('D')).mean().asfreq('D')
    print(f'{city} (walk-forward): autocorrelation of the daily error difference at lag 1/2/3 days = '
          + ' / '.join(f'{daily_diff.autocorr(k):.2f}' for k in (1, 2, 3)))


def block_bootstrap_gain(p, model_a, model_b, block_days=BLOCK_DAYS, n_boot=N_BOOT, seed=RANDOM_STATE):
    """Paired block bootstrap of the MAE gain of model_a over model_b, i.e. 1 - MAE_a / MAE_b.

    p: DataFrame indexed by issue time t with an 'actual' column and one column per model.
    Samples are grouped into consecutive calendar blocks of `block_days` days (forecast time t + HORIZON);
    whole blocks are resampled with replacement, keeping the autocorrelation of the errors inside each block.
    """
    t = p.index + pd.Timedelta(hours=HORIZON)
    codes = pd.factorize(np.asarray((t - t.min()) // pd.Timedelta(days=block_days)))[0]
    abs_err_a = np.bincount(codes, weights=np.abs(p[model_a] - p['actual']))   # error sum per block
    abs_err_b = np.bincount(codes, weights=np.abs(p[model_b] - p['actual']))
    k = len(abs_err_a)
    counts = np.random.default_rng(seed).multinomial(k, np.full(k, 1 / k), size=n_boot)  # how often each block is drawn
    gains = 1 - (counts @ abs_err_a) / (counts @ abs_err_b)
    low, high = np.percentile(gains, [2.5, 97.5])
    return {'gain': 1 - abs_err_a.sum() / abs_err_b.sum(), 'ci_low': low, 'ci_high': high, 'blocks': k}


def competitors(city):
    return ['Persistence (baseline)', 'Naive blend (baseline)'] + [
        m for m in ['Ridge', 'RandomForest', 'HistGradientBoosting'] if m != best_model[city]]


rows = []
for city, f in forecasters.items():
    for set_name, preds in [('Walk-forward', f.wf_predictions), ('Test', f.test_predictions)]:
        for other in competitors(city):
            rows.append({'city': city, 'set': set_name, 'model': best_model[city], 'vs': other,
                         **block_bootstrap_gain(preds, best_model[city], other)})
boot = pd.DataFrame(rows)
boot['CI excludes 0'] = (boot['ci_low'] > 0) | (boot['ci_high'] < 0)
display(boot.style.format({'gain': '{:+.1%}', 'ci_low': '{:+.1%}', 'ci_high': '{:+.1%}'}))
for r in boot.itertuples():
    print(f'BOOT | {r.city} | {r.set} | {r.model} vs {r.vs} | gain {r.gain:+.1%} [{r.ci_low:+.1%}, {r.ci_high:+.1%}]')

print('\nSensitivity to the block length (best model vs Naive blend):')
for city, f in forecasters.items():
    for set_name, preds in [('Walk-forward', f.wf_predictions), ('Test', f.test_predictions)]:
        cis = {L: block_bootstrap_gain(preds, best_model[city], 'Naive blend (baseline)', block_days=L)
               for L in (1, 3, 7)}
        print(f'{city} {set_name}: ' + ' | '.join(f"{L}-day blocks ({c['blocks']}): [{c['ci_low']:+.1%}, {c['ci_high']:+.1%}]"
                                                  for L, c in cis.items()))

**Figure 7 – Gain of the best model of each city over every competitor**, with 95% block-bootstrap confidence intervals,
on the walk-forward forecasts (hollow) and Test (filled). A bar that crosses the dashed zero line means the difference could be luck.

In [ ]:
# Forest plot of the bootstrap confidence intervals
SET_STYLE = {'Walk-forward': {'color': '#eb6834', 'mfc': 'white', 'offset': -0.17},   # y axis is inverted: walk-forward on top
             'Test': {'color': '#2a78d6', 'mfc': '#2a78d6', 'offset': 0.17}}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharex=True)
for ax, city in zip(axes, CITY_FILES):
    labels = competitors(city)
    for set_name, st in SET_STYLE.items():
        sub = boot[(boot.city == city) & (boot.set == set_name)].set_index('vs').loc[labels]
        y = np.arange(len(labels)) + st['offset']
        ax.hlines(y, sub['ci_low'] * 100, sub['ci_high'] * 100, color=st['color'], lw=2)
        ax.plot(sub['gain'] * 100, y, 'o', ms=8, color=st['color'], mfc=st['mfc'], mew=2, label=set_name)
        if set_name == 'Test':
            for yi, g in zip(y, sub['gain']):
                ax.annotate(f'{g:+.1%}', (g * 100, yi), xytext=(0, -13), textcoords='offset points',
                            ha='center', fontsize=9, color='#333333')
    ax.axvline(0, color='#555555', ls='--', lw=1)
    ax.set_yticks(range(len(labels)), [f'vs {l}' for l in labels])
    ax.invert_yaxis()
    ax.set(title=f'{city} – {best_model[city]}', xlabel='MAE reduction (%), 95% block-bootstrap CI')
    ax.grid(axis='y', visible=False)
axes[0].legend(loc='lower right')
plt.tight_layout()
plt.show()

**Figure 7 caption — which differences are real?** (95% CI, 3-day blocks; *significant* = the interval excludes 0)

| City | Comparison | Walk-forward (10/2025–08/2026) | Test (09–11/2026) |
|---|---|---|---|
| Hanoi | RandomForest vs Persistence | **+9.7% [+2.0, +16.5]** | **+9.6% [+1.5, +17.3]** |
| Hanoi | RandomForest vs Naive blend | +6.8% [−0.9, +13.7] | +5.3% [−3.4, +13.3] |
| Hanoi | RandomForest vs Ridge | +0.5% [−2.5, +3.8] | −3.4% [−9.3, +1.6] |
| HCMC | HGB vs Persistence | **+12.0% [+7.4, +15.8]** | **+14.1% [+8.7, +19.4]** |
| HCMC | HGB vs Naive blend | +3.3% [−1.4, +7.6] | **+7.2% [+1.7, +12.6]** |
| HCMC | HGB vs RandomForest | **+1.6% [+0.3, +3.0]** | −2.2% [−5.0, +0.5] |

- **Against Persistence**, the best model is significantly better in both cities, on both evaluation sets.
- **Against the Naive blend (the pre-specified comparisons):**
  - **HCMC:** significant on Test (+7.2%, CI +1.7% to +12.6%, positive for 1-, 3- and 7-day blocks), not significant on walk-forward (+3.3%).
  - **Hanoi:** not significant on Test (+5.3%, CI −3.4% to +13.3%). On walk-forward the result is borderline: the interval reaches just below 0 for 1- and
    3-day blocks and stays above 0 for 7-day blocks. We therefore do **not** claim a significant gain in Hanoi.
  - The point estimates are positive everywhere, so the most likely situation is a small real gain (a few %) that one year of data cannot pin down.
- **Between the ML models**, almost no difference is significant, and the only one (HCMC, HGB vs RandomForest on walk-forward, +1.6%) reverses sign on Test.
  *Which ML model is best is not established in either city.*
- **Block length:** the autocorrelation of the daily error difference is small (Hanoi 0.15 / −0.06 / 0.02 at 1 / 2 / 3 days; HCMC 0.16 / 0.20 / 0.10),
  and the conclusions above do not change between 1-, 3- and 7-day blocks except for the borderline Hanoi walk-forward case.
- **Method note:** an i.i.d. bootstrap over single hours would give much narrower intervals and declare almost everything significant;
  the blocks keep the strong hour-to-hour correlation of forecast errors.

### 4.4. Forecast vs. actual
**Figure 8 – First 14 days of the Test set:** actual values, the Naive blend baseline and the best model of each city (chosen by walk-forward validation)
(the time axis is the **forecast** time, i.e. $t + 24h$).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for ax, (city, f) in zip(axes, forecasters.items()):
    p = f.test_predictions
    w = p.loc[: p.index.min() + pd.Timedelta(days=14)]
    target_time = w.index + pd.Timedelta(hours=HORIZON)
    ax.plot(target_time, w['actual'], color='k', lw=1.3, label='Actual')
    ax.plot(target_time, w['Naive blend (baseline)'], color='gray', ls=':', label='Naive blend (baseline)')
    ax.plot(target_time, w[best_model[city]], color=CITY_COLORS[city], label=best_model[city])
    mae_blend = mean_absolute_error(w['actual'], w['Naive blend (baseline)'])
    mae_best = mean_absolute_error(w['actual'], w[best_model[city]])
    print(f'{city} – first 14 days of Test ({len(w)} hours): MAE Naive blend {mae_blend:.2f} vs {best_model[city]} {mae_best:.2f} | '
          f'std of forecast {w[best_model[city]].std():.1f} vs actual {w["actual"].std():.1f} µg/m³')
    ax.set(title=f'{city} – 24h-ahead forecast (MAE in this window: Naive blend {mae_blend:.2f}, {best_model[city]} {mae_best:.2f})',
           ylabel='PM2.5 (µg/m³)')
    ax.legend()
plt.tight_layout()
plt.show()

**Figure 8 caption:** In this window the Hanoi model follows the actual series better than the Naive blend (MAE 10.99 vs 12.42 µg/m³ over 337 hours),
while in HCMC it is slightly worse (11.45 vs 10.85): two weeks are too short to judge a model, which is why Section 4.3 uses the whole sets.
In both cities the forecast is much smoother than reality: its standard deviation is less than half of the actual one
(Hanoi 7.2 vs 15.5, HCMC 5.5 vs 18.5 µg/m³), so short spikes are missed. This is the visual counterpart of the under-forecasting of polluted hours in Section 4.5.

### 4.5. Error analysis
A regression problem has no direct confusion matrix, so we (1) analyze the error by **hour of day** (the error by **month** is the
walk-forward figure of Section 3.2), and
(2) put both the actual and the forecast PM2.5 into **concentration bands** defined by the US EPA 2024 PM2.5 breakpoints
(Good ≤ 9.0, Moderate 9.1–35.4, USG 35.5–55.4, Unhealthy 55.5–125.4, Very unhealthy and above ≥ 125.5 µg/m³) and build a
**band confusion matrix** — showing which band the model tends to confuse with which.

*Precision about the definition:* the official EPA AQI is computed from a **24-hour average** (or the NowCast for hourly reporting).
Here the breakpoints are applied to **hourly** concentrations, so these are "EPA concentration bands", **not** official AQI values.
As in the EPA rule, a concentration is first truncated to one decimal (e.g. 9.05 → 9.0 → Good) and then binned.

In [ ]:
MIN_SAMPLES = 20   # hours of the day with fewer samples are not shown (too noisy)
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, (city, f) in zip(axes, forecasters.items()):
    for set_name, p, style in [('Walk-forward', f.wf_predictions, dict(ls='--', marker='o', mfc='white')),
                               ('Test', f.test_predictions, dict(ls='-', marker='o'))]:
        err = (p[best_model[city]] - p['actual']).abs()
        t = p.index + pd.Timedelta(hours=HORIZON)                  # forecast time
        hour_stats = err.groupby(t.hour).agg(['mean', 'size'])
        by_hour = hour_stats.loc[hour_stats['size'] >= MIN_SAMPLES, 'mean']
        ax.plot(by_hour.index, by_hour, color=CITY_COLORS[city], label=set_name, **style)
        print(f'{city} {set_name}: MAE by hour of the forecast time min {by_hour.min():.1f} (h{by_hour.idxmin()}) – '
              f'max {by_hour.max():.1f} (h{by_hour.idxmax()}) | hours with < {MIN_SAMPLES} samples (not shown): '
              f'{hour_stats.index[hour_stats["size"] < MIN_SAMPLES].tolist()}')
    ax.set(title=f'{city} – MAE by hour of day ({best_model[city]})', xlabel='Hour of the forecast time',
           ylabel='MAE (µg/m³)', xticks=range(0, 24, 3))
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
EPA_BINS = [-np.inf, 9.0, 35.4, 55.4, 125.4, np.inf]      # right-closed: (-inf, 9.0], (9.0, 35.4], ...
EPA_LABELS = ['Good', 'Moderate', 'USG', 'Unhealthy', 'Very unhealthy+']


def to_epa_band(values):
    """Truncate to 0.1 µg/m³ (EPA rule), then map each hourly PM2.5 value to its EPA concentration band."""
    truncated = np.floor(np.asarray(values, dtype=float) * 10) / 10
    return pd.cut(pd.Series(truncated), bins=EPA_BINS, labels=EPA_LABELS)


fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (city, f) in zip(axes, forecasters.items()):
    wf = f.wf_predictions
    wf_acc = (to_epa_band(wf['actual']) == to_epa_band(wf[best_model[city]])).mean()
    print(f'{city}: correct EPA band on the walk-forward forecasts {wf_acc:.1%} (for comparison with Test below)')
    p = f.test_predictions
    actual_lvl, pred_lvl = to_epa_band(p['actual']), to_epa_band(p[best_model[city]])
    cm = pd.crosstab(actual_lvl, pred_lvl, rownames=['Actual'], colnames=['Predicted'], dropna=False)
    cm = cm.reindex(index=EPA_LABELS, columns=EPA_LABELS, fill_value=0)
    cm_norm = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    sns.heatmap(cm_norm, annot=cm.values, fmt='d', cmap='Blues', cbar=False, ax=ax)
    acc = (actual_lvl == pred_lvl).mean()
    ax.set_title(f'{city} – {best_model[city]} (Test) | correct EPA band: {acc:.1%}')
    print(f'{city} Test: correct EPA band {acc:.1%} | within ±1 band: '
          f'{(abs(actual_lvl.cat.codes - pred_lvl.cat.codes) <= 1).mean():.1%}')
    print('Measured hours per band:', cm.sum(axis=1).to_dict())
    print(cm, end='\n\n')
plt.tight_layout()
plt.show()

*USG = Unhealthy for Sensitive Groups (35.5–55.4 µg/m³).*


**Error analysis findings:**
- **By hour:** the error is highest for forecasts of the early morning (Hanoi 6h: 13.8 µg/m³ on both sets; HCMC walk-forward 7h: 11.1) and lowest
  in the late afternoon or evening (Hanoi 17h: 10.6 walk-forward, 8.2 Test; HCMC 19h–20h: 7.5 / 8.5), i.e. where the night-time accumulation and the
  morning rush hour make the level change fastest. The error by month is the walk-forward figure of Section 3.2.
- **By EPA concentration band (Test):** the correct band is predicted for 59.3% (Hanoi) and 60.5% (HCMC) of hours, 96.7% / 96.8% within ±1 band
  (walk-forward: 61.0% / 66.9%). The main error is **under-forecasting polluted hours**: in Hanoi only 46 of the 215 "Unhealthy" hours are predicted
  correctly and 271 of the 455 "USG" hours are predicted as "Moderate"; in HCMC only 2 of the 136 "Unhealthy" hours are correct.
  No model forecast is ever "Good" (the 132 / 37 "Good" hours are all over-forecast): the forecasts are pulled towards the mean,
  the expected behaviour of a model trained to minimize MAE on a noisy target.
- **Practical consequence:** the model is useful for following the general level, but **not reliable as an early-warning tool for severe pollution episodes**,
  where a missed alarm costs more than a false alarm.

### 4.6. Feature importance (permutation importance)
Measures how much the Test MAE increases when each original feature of the best model is randomly shuffled.
*Caveat:* the PM2.5 lag/rolling features are strongly correlated, so shuffling one of them hurts less than its real
contribution (the others carry the same information) — the importance of the "PM2.5 history" group is spread over several features.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (city, f) in zip(axes, forecasters.items()):
    model = f.models[best_model[city]]
    imp = permutation_importance(model, f.test[FEATURES], f.test[TARGET], n_repeats=5,
                                 scoring='neg_mean_absolute_error', random_state=RANDOM_STATE)
    s = pd.Series(imp.importances_mean, index=FEATURES).nlargest(10).sort_values()
    s.plot.barh(ax=ax, color=CITY_COLORS[city])
    print(f'{city} – top 5:', s[::-1].head(5).round(2).to_dict())
    ax.set(title=f'{city} – Top 10 features ({best_model[city]})', xlabel='MAE increase (µg/m³)')
plt.tight_layout()
plt.show()

Because single-feature importance is diluted by correlated features, we also shuffle **whole groups** of features together
(grouped permutation importance): PM2.5 history, weather (including wind sector) and calendar (hour, season, day of week).

In [ ]:
FEATURE_GROUPS = {'PM2.5 history': PM_FEATURES,
                  'Weather': WEATHER_FEATURES + ['wind_sector'],
                  'Calendar': CYCLIC_FEATURES + ['season', 'day_of_week']}


def group_importance(model, X, y, groups, n_repeats=10, seed=RANDOM_STATE):
    """Mean MAE increase when all columns of a group are shuffled together (same row permutation)."""
    rng = np.random.default_rng(seed)
    base = mean_absolute_error(y, model.predict(X))
    result = {}
    for name, cols in groups.items():
        deltas = []
        for _ in range(n_repeats):
            Xs = X.copy()
            Xs[cols] = X[cols].values[rng.permutation(len(X))]
            deltas.append(mean_absolute_error(y, model.predict(Xs)) - base)
        result[name] = np.mean(deltas)
    return result


group_imp = pd.DataFrame({city: group_importance(f.models[best_model[city]], f.test[FEATURES], f.test[TARGET], FEATURE_GROUPS)
                          for city, f in forecasters.items()})
print('Grouped importance (MAE increase, µg/m³):', group_imp.round(2).to_dict())
display(group_imp.round(2))

**Findings (Test set):**
- **Hanoi (RandomForest):** the current PM2.5 value dominates (`pm25_lag_0`: +2.37 µg/m³ when shuffled); the next features are weather
  (surface pressure +0.17, 24h rain +0.17, wind speed +0.12). By group: **PM2.5 history +3.82, weather +0.95, calendar −0.06**.
  A negative value means shuffling the calendar features made the error slightly *smaller*: they carry no signal on this period.
- **HCMC (HistGradientBoosting):** the current value is still first (+0.64), followed by the **hour of day** (`hour_cos` +0.24) and 24h rain (+0.10).
  By group: **PM2.5 history +1.55, calendar +0.26, weather +0.17** — HCMC's PM2.5 follows the daily traffic rhythm (EDA: 7h peak).

The drivers differ between the cities (Hanoi: recent level + weather; HCMC: recent level + time of day), which supports training one model per city.

### 4.7. Inference demo: `predict_next`
Takes the hourly observation history up to time $t$ and returns the PM2.5 forecast at $t + 24h$ together with its EPA concentration band.
Demo: we use a history ending 24 hours before the last available observation, then compare with the known actual value.

In [ ]:
def predict_next(city, history):
    """history: hourly DataFrame (same columns as the cleaned data), ending at time t."""
    model = forecasters[city].models[best_model[city]]
    x_last = build_features(history, city).iloc[[-1]]
    value = float(model.predict(x_last[FEATURES])[0])
    return {'city': city, 'model': best_model[city],
            'forecast_time': x_last.index[0] + pd.Timedelta(hours=HORIZON),
            'pm25_forecast': round(value, 1), 'epa_band': str(to_epa_band([value])[0])}


demo_rows = []
for city, df in clean.items():
    t_end = df['pm25_raw'].last_valid_index() - pd.Timedelta(hours=HORIZON)
    history = df.loc[t_end - pd.Timedelta(days=7): t_end]
    res = predict_next(city, history)
    actual = df.loc[res['forecast_time'], 'pm25_raw']
    res.update({'pm25_actual': round(float(actual), 1), 'epa_band_actual': str(to_epa_band([actual])[0])})
    demo_rows.append(res)
display(pd.DataFrame(demo_rows))

### 4.8. Conclusions, limitations and future work


**Conclusions:**
- We built a 24h-ahead PM2.5 forecasting pipeline, applied identically but **trained independently** for Hanoi and HCMC, using reference-station
  (Hanoi) and low-cost-sensor (HCMC) data from OpenAQ combined with Open-Meteo weather, over the same period for both cities (04/07/2025 – 14/11/2026).
  Features use only data available at forecast time (causal forward-fill), and models are evaluated only on measured values.
- Models are selected by **walk-forward validation** over 11 months that include the whole winter, and evaluated once on Test (09–11/2026).
- The best model lowers MAE by **5.3% in Hanoi (RandomForest)** and **7.2% in HCMC (HistGradientBoosting)** on Test compared with the
  strongest naive baseline (Naive blend), and by 9.6% / 14.1% compared with Persistence.
- Block-bootstrap intervals: the gain over Persistence is significant everywhere; the gain over the Naive blend is significant only for HCMC on Test;
  which ML model is best is not established in either city.
- The ML gain is concentrated in the stable winter peak; in months where the pollution level changes (Hanoi November and April, HCMC October and February)
  the ML models lag behind and can be worse than the Naive blend.

**Limitations:**
- **One winter only** (the Hanoi station started in 07/2025): the models never see two winters, and the walk-forward winter months are the only evidence
  about winter performance. The Test period covers the start of the polluted season, not its peak.
- One station per city; the HCMC station is a low-cost sensor (AirGradient), possibly less accurate at high humidity, and the only HCMC station with continuous data.
- Hyperparameters were tuned on the whole Development set, so the walk-forward scores of the ML models are slightly optimistic; several selected values lie at the
  edge of the grid. The Test scores are not affected.
- Weather features use observations at $t$, not weather forecasts for $t+24h$. The models under-forecast high-pollution episodes.

**Future work:**
- Re-run the walk-forward evaluation after a second winter (from 12/2026) to check whether the winter gain repeats.
- Add **weather forecasts** for the target time and more stations per city; add features that react to level changes (e.g. the change of the 24h mean over several days).
- Use quantile regression or sample weights that emphasize high-pollution hours, or a threshold classifier for early warning.
- Re-tune hyperparameters inside each walk-forward fold (nested tuning) to remove the optimism of the walk-forward scores.